# Bayesian Model Averaging of Glacier Contribution to Sea-Level Rise (2026–2056)

**Task:** Combine four statistical models (Linear Regression, Polynomial Regression, ARIMA, Bayesian Linear Regression) with Bayesian Model Averaging (BMA) and quantify the uncertainty in the glacier contribution to sea-level rise over the next 30 years.

**Pipeline**
1. Load the World Glacier Inventory (`database.csv`) and estimate the total ice volume that is available to melt (sea-level equivalent, SLE).
2. Build an annual time series of glacier sea-level contribution (mm/yr).
3. Fit the four models; each produces **probabilistic** forecasts (predictive samples) for 2026–2056.
4. Compute BMA weights from rolling-origin out-of-sample predictive performance.
5. Mix the models, then split the total uncertainty into *within-model* and *between-model* parts.

> ▶️ **Run the cells in order, top to bottom** (Colab: *Runtime → Run all*, Jupyter: *Run → Run All Cells*). Running a later cell first causes `NameError` / `FileNotFoundError`.

> ⚠️ **Important data note.** `database.csv` is a *static inventory* (area, elevation, depth of each glacier). It contains **no time series of melt**. Regression / ARIMA models need a time series, so Section 2 uses a **synthetic placeholder series**, shaped after published global glacier mass-loss rates (≈ 0.65–0.85 mm/yr SLE around 2000–2025). Set `REAL_DATA_PATH` to a CSV of real observations (e.g. WGMS or Hugonnet et al. 2021 regional sums) to replace it. The inventory is used for the ice-volume ceiling and for sanity checks.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ---------------- Configuration ----------------
INVENTORY_PATH = "database.csv"
REAL_DATA_PATH = None        # CSV with columns: year, mass_loss_gt  (positive = ice lost). None -> synthetic series
SEED           = 42
HIST_START, HIST_END = 2000, 2025
FC_START,   FC_END   = 2026, 2056
N_SAMPLES      = 4000        # predictive samples per model
CV_HORIZON     = 3           # years ahead in rolling-origin validation
MIN_TRAIN      = 14          # minimum training years in validation
POLY_DEGREE    = 2
WEIGHT_METHOD  = "pseudo_bma"  # "pseudo_bma" or "em"
GT_PER_MM_SLE  = 362.5       # Gt of water per mm of global sea level
ICE_DENSITY    = 0.917       # Gt per km^3 of ice

rng = np.random.default_rng(SEED)

## 0. Upload the dataset
Run the next cell and choose `database.csv` when prompted. In **Google Colab** a file-picker button appears. In **Jupyter / VS Code** an upload widget appears: click it, pick the file, then run the cell after it. If `database.csv` is already in the notebook's folder, this step is skipped.

In [ ]:
import os
INVENTORY_PATH = "database.csv"     # where the dataset will be saved / read from

try:
    from google.colab import files      # only exists inside Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import glob, shutil

def find_inventory():
    # Look for database*.csv in the current folder and common download locations
    roots = [".", os.path.expanduser("~/Downloads"), os.path.expanduser("~/Desktop"),
             "/content", "/content/drive/MyDrive", "/mnt/data"]
    for r in roots:
        hits = glob.glob(os.path.join(r, "database*.csv"))
        if hits:
            return hits[0]
    return None

if not os.path.exists(INVENTORY_PATH):
    found = find_inventory()
    if found:
        shutil.copy(found, INVENTORY_PATH)
        print(f"Copied {found} -> {INVENTORY_PATH}")

uploader = None
if os.path.exists(INVENTORY_PATH):
    print(f"Found {INVENTORY_PATH} ({os.path.getsize(INVENTORY_PATH)/1e6:.1f} MB) - no upload needed.")
elif IN_COLAB:
    uploaded = files.upload()           # opens the file picker
    name = next(iter(uploaded))
    if name != INVENTORY_PATH:
        os.replace(name, INVENTORY_PATH)
    print(f"Saved as {INVENTORY_PATH}")
else:
    try:
        import ipywidgets as widgets
    except ImportError:
        raise SystemExit("Run `pip install ipywidgets` (or put database.csv next to this notebook) and rerun.")
    uploader = widgets.FileUpload(accept=".csv", multiple=False, description="Upload database.csv")
    display(uploader)
    print("Click the button, choose database.csv, then run the next cell.")

In [ ]:
# Finishes the upload when using the Jupyter / VS Code widget (does nothing in Colab or if the file already exists)
import os
INVENTORY_PATH = "database.csv"
if not os.path.exists(INVENTORY_PATH):
    if uploader is None or not uploader.value:
        raise FileNotFoundError("No file uploaded yet - use the upload button above, then rerun this cell.")
    val = uploader.value
    item = val[0] if isinstance(val, (tuple, list)) else next(iter(val.values()))   # ipywidgets 8 vs 7
    with open(INVENTORY_PATH, "wb") as f:
        f.write(bytes(item["content"]))
    print(f"Saved {INVENTORY_PATH} ({os.path.getsize(INVENTORY_PATH)/1e6:.1f} MB)")
else:
    print(f"{INVENTORY_PATH} is ready.")

## 1. Inventory: how much ice is there?
Ice volume per glacier is estimated with volume–area scaling, V = 0.034·A^1.375 (km³, A in km²; Bahr et al. 1997), and compared with an estimate from the recorded mean depth where it exists.

In [ ]:
if not os.path.exists(INVENTORY_PATH):
    raise FileNotFoundError("database.csv is not here yet. Run the cells in section 0 (upload) first, "
                            "and wait until it prints 'Saved' / 'Found' / 'is ready'.")
if not os.path.exists(INVENTORY_PATH):
    raise FileNotFoundError(
        f"{INVENTORY_PATH} not found. Run the upload cells in Section 0 first (and choose database.csv).")
inv = pd.read_csv(INVENTORY_PATH, low_memory=False)
print("Raw rows:", len(inv))

inv = inv[inv["Glacier Area"].notna() & (inv["Glacier Area"] > 0)].copy()
print("Rows with valid area:", len(inv))

inv["vol_scaling_km3"] = 0.034 * inv["Glacier Area"] ** 1.375
inv["vol_depth_km3"]   = inv["Glacier Area"] * inv["Mean Depth"] / 1000.0   # km2 * m -> km3

def to_sle_mm(vol_km3):
    return vol_km3.sum() * ICE_DENSITY / GT_PER_MM_SLE

total_scaling = to_sle_mm(inv["vol_scaling_km3"])
has_depth = inv["Mean Depth"].notna()
print(f"Total area: {inv['Glacier Area'].sum():,.0f} km2")
print(f"Ice volume ceiling (scaling): {inv['vol_scaling_km3'].sum():,.0f} km3 -> {total_scaling:,.0f} mm SLE")
print(f"Glaciers with recorded depth: {has_depth.mean():.1%}")
print("On that subset: scaling =", round(to_sle_mm(inv.loc[has_depth,'vol_scaling_km3']),1),
      "mm SLE | depth-based =", round(to_sle_mm(inv.loc[has_depth,'vol_depth_km3']),1), "mm SLE")

by_cont = (inv.groupby("Continent")
              .agg(glaciers=("Glacier ID","count"), area_km2=("Glacier Area","sum"),
                   vol_km3=("vol_scaling_km3","sum")))
by_cont["SLE_mm"] = by_cont["vol_km3"] * ICE_DENSITY / GT_PER_MM_SLE
by_cont.sort_values("area_km2", ascending=False).round(1)

## 2. Time series of glacier sea-level contribution
Target variable: **annual SLE contribution (mm/yr)**, always ≥ 0 for net-melting glaciers.

In [ ]:
years = np.arange(HIST_START, HIST_END + 1)

if REAL_DATA_PATH:
    obs = pd.read_csv(REAL_DATA_PATH).sort_values("year")
    years = obs["year"].to_numpy()
    y = obs["mass_loss_gt"].to_numpy() / GT_PER_MM_SLE
    SERIES_LABEL = "observed"
else:
    # SYNTHETIC placeholder: accelerating loss (~230 -> ~305 Gt/yr) + AR(1) noise
    n = len(years)
    trend = 230 + 3.0 * (years - HIST_START) + 0.03 * (years - HIST_START) ** 2
    noise = np.zeros(n)
    for i in range(1, n):
        noise[i] = 0.4 * noise[i-1] + rng.normal(0, 28)
    y = np.clip(trend + noise, 50, None) / GT_PER_MM_SLE
    SERIES_LABEL = "synthetic placeholder"

fc_years = np.arange(FC_START, FC_END + 1)
t_hist = years.astype(float)
t_fc   = fc_years.astype(float)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(years, y, "o-", color="k")
ax.set(title=f"Annual glacier SLE contribution ({SERIES_LABEL})", xlabel="Year", ylabel="mm / yr")
plt.show()
print(f"{len(y)} annual values, mean {y.mean():.2f} mm/yr")

## 3. The four probabilistic models

Every model exposes the same interface:

* `fit(t, y)`
* `predictive(t_new)` → mean, standard deviation and degrees of freedom of the one-step marginal predictive distribution (used to score the model in validation)
* `sample_paths(t_new, n, rng)` → `n` joint predictive trajectories (used for the 2026–2056 forecast)

| Model | Uncertainty source |
|---|---|
| Linear Regression (OLS) | parameter uncertainty + residual noise (Student-t predictive) |
| Polynomial Regression (OLS, degree 2) | same, with a quadratic trend |
| ARIMA(1,1,1) with drift | forecast error accumulates with horizon (parameters fixed at their estimates) |
| Bayesian Linear Regression | conjugate Normal–Inverse-Gamma posterior; the prior shrinks the trend |

In [ ]:
def logpdf(yv, mean, sd, df):
    if np.isinf(df):
        return stats.norm.logpdf(yv, mean, sd)
    return stats.t.logpdf(yv, df, mean, sd)


class PolyOLS:
    """Ordinary least squares polynomial regression (degree=1 -> linear regression)."""
    def __init__(self, degree=1, name=None):
        self.degree = degree
        self.name = name or ("Linear Regression" if degree == 1 else f"Polynomial Regression (deg {degree})")

    def _X(self, t):
        return np.vander((np.asarray(t, float) - self.t0) / self.sc, self.degree + 1, increasing=True)

    def fit(self, t, y):
        t = np.asarray(t, float)
        self.t0, self.sc = t.mean(), t.std()
        X = self._X(t); n, self.p = X.shape
        self.XtXi = np.linalg.inv(X.T @ X)
        self.beta = self.XtXi @ X.T @ y
        res = y - X @ self.beta
        self.dof = n - self.p
        self.s2 = float(res @ res / self.dof)
        return self

    def predictive(self, t_new):
        X = self._X(t_new)
        mean = X @ self.beta
        sd = np.sqrt(self.s2 * (1 + np.einsum("ij,jk,ik->i", X, self.XtXi, X)))
        return mean, sd, self.dof

    def sample_paths(self, t_new, n, rng):
        X = self._X(t_new)
        sig2 = self.s2 * self.dof / rng.chisquare(self.dof, size=n)      # sigma^2 | data
        L = np.linalg.cholesky(self.XtXi)
        z = rng.standard_normal((n, self.p))
        betas = self.beta + (np.sqrt(sig2)[:, None] * z) @ L.T            # beta | sigma^2, data
        return betas @ X.T + np.sqrt(sig2)[:, None] * rng.standard_normal((n, len(t_new)))


class BayesLinReg:
    """Conjugate Bayesian linear regression, beta|s2 ~ N(0, s2*tau2*I), s2 ~ InvGamma(a0, b0).
    Time and target are standardised so the priors are on a comparable scale."""
    name = "Bayesian Linear Regression"
    def __init__(self, tau2=1.0, a0=2.0, b0=0.1):
        self.tau2, self.a0, self.b0 = tau2, a0, b0

    def _X(self, t):
        z = (np.asarray(t, float) - self.t0) / self.sc
        return np.column_stack([np.ones_like(z), z])

    def fit(self, t, y):
        t = np.asarray(t, float)
        self.t0, self.sc = t.mean(), t.std()
        self.ym, self.ys = y.mean(), y.std()
        ys_ = (y - self.ym) / self.ys
        X = self._X(t); n = len(y)
        P0 = np.eye(2) / self.tau2
        self.Vn = np.linalg.inv(P0 + X.T @ X)
        self.mn = self.Vn @ (X.T @ ys_)
        self.an = self.a0 + n / 2
        self.bn = self.b0 + 0.5 * (ys_ @ ys_ - self.mn @ np.linalg.inv(self.Vn) @ self.mn)
        return self

    def predictive(self, t_new):
        X = self._X(t_new)
        mean = X @ self.mn
        scale = np.sqrt(self.bn / self.an * (1 + np.einsum("ij,jk,ik->i", X, self.Vn, X)))
        return self.ym + self.ys * mean, self.ys * scale, 2 * self.an

    def sample_paths(self, t_new, n, rng):
        X = self._X(t_new)
        sig2 = self.bn / rng.gamma(self.an, size=n)
        L = np.linalg.cholesky(self.Vn)
        z = rng.standard_normal((n, 2))
        betas = self.mn + (np.sqrt(sig2)[:, None] * z) @ L.T
        draws = betas @ X.T + np.sqrt(sig2)[:, None] * rng.standard_normal((n, len(t_new)))
        return self.ym + self.ys * draws


class ARIMAModel:
    """ARIMA(p,d,q) with drift. Predictive distribution from the state-space forecast."""
    def __init__(self, order=(1, 1, 1)):
        self.order = order
        self.name = f"ARIMA{order}"

    def fit(self, t, y):
        self.res = ARIMA(np.asarray(y, float), order=self.order, trend="t").fit()
        self.last_t = float(np.asarray(t)[-1])
        return self

    def predictive(self, t_new):
        h = int(round(float(np.asarray(t_new)[-1]) - self.last_t))
        fc = self.res.get_forecast(steps=h)
        return (np.atleast_1d(fc.predicted_mean)[-1:], np.atleast_1d(fc.se_mean)[-1:], np.inf)

    def sample_paths(self, t_new, n, rng):
        seed = int(rng.integers(1_000_000))
        try:      # statsmodels >= 0.15
            sims = self.res.simulate(nsimulations=len(t_new), repetitions=n, anchor="end",
                                     rng=np.random.default_rng(seed))
        except TypeError:   # older statsmodels
            sims = self.res.simulate(nsimulations=len(t_new), repetitions=n, anchor="end",
                                     random_state=seed)
        return np.asarray(sims).reshape(len(t_new), n).T


MODEL_FACTORIES = {
    "Linear Regression":          lambda: PolyOLS(1),
    "Polynomial Regression":      lambda: PolyOLS(POLY_DEGREE),
    "ARIMA(1,1,1)":               lambda: ARIMAModel((1, 1, 1)),
    "Bayesian Linear Regression": lambda: BayesLinReg(),
}
print("Models:", list(MODEL_FACTORIES))

## 4. BMA weights from rolling-origin validation

Exact marginal likelihoods are not available for all four models (e.g. ARIMA), so the posterior model probability is approximated with the **out-of-sample predictive likelihood**:

$$ w_k \;\propto\; \prod_{t} p_k\big(y_{t+h}\mid y_{1:t}\big) \quad\text{(uniform prior over models)} $$

For each origin the model is refit on the past only and scores the observation `CV_HORIZON` years ahead. Two weighting rules are computed: the product above (`pseudo_bma`) and the EM mixture weights of Raftery et al. (2005) (`em`), which are more stable when one model dominates.

In [ ]:
def rolling_logscores(factory, t, y, h, min_train):
    out, targets = [], []
    for end in range(min_train, len(y) - h + 1):
        m = factory().fit(t[:end], y[:end])
        tgt = end + h - 1
        mean, sd, df = m.predictive(t[[tgt]])
        out.append(float(logpdf(y[tgt], mean[0], sd[0], df)))
        targets.append(int(t[tgt]))
    return np.array(out), targets

logS = {}
for name, fac in MODEL_FACTORIES.items():
    logS[name], target_years = rolling_logscores(fac, t_hist, y, CV_HORIZON, MIN_TRAIN)
L = pd.DataFrame(logS, index=target_years)
print(f"Validation targets: {target_years[0]}–{target_years[-1]}  ({len(L)} forecasts, horizon {CV_HORIZON} yr)")

# --- pseudo-BMA weights
tot = L.sum()
w_pseudo = np.exp(tot - tot.max()); w_pseudo /= w_pseudo.sum()

# --- EM weights (mixture of predictive densities)
dens = np.exp(L.values)                   # (T, K)
w = np.ones(dens.shape[1]) / dens.shape[1]
for _ in range(2000):
    z = dens * w
    z /= z.sum(axis=1, keepdims=True)
    w_new = z.mean(axis=0)
    if np.abs(w_new - w).max() < 1e-10:
        break
    w = w_new
w_em = pd.Series(w, index=L.columns)

weights = pd.DataFrame({"total log score": tot, "pseudo_bma": w_pseudo, "em": w_em}).round(4)
display(weights)

W = (w_pseudo if WEIGHT_METHOD == "pseudo_bma" else w_em).copy()
W = W / W.sum()
print("Weights used (", WEIGHT_METHOD, "):", W.round(3).to_dict())

ax = weights[["pseudo_bma", "em"]].plot.bar(figsize=(8, 3.5), rot=15)
ax.set(title="BMA model weights", ylabel="weight"); plt.show()

## 5. Final fit and probabilistic forecasts for 2026–2056
Each model is refit on all data and produces `N_SAMPLES` joint predictive trajectories. Trajectories are clipped at 0 because a glacier contribution to sea level cannot be negative in this setup.

In [ ]:
paths = {}
for name, fac in MODEL_FACTORIES.items():
    m = fac().fit(t_hist, y)
    paths[name] = np.clip(m.sample_paths(t_fc, N_SAMPLES, rng), 0, None)

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, (name, P) in zip(axes.ravel(), paths.items()):
    lo, med, hi = np.percentile(P, [5, 50, 95], axis=0)
    ax.plot(years, y, "ko-", ms=3)
    ax.plot(fc_years, med, color="C0"); ax.fill_between(fc_years, lo, hi, alpha=0.25, color="C0")
    ax.set_title(f"{name}  (w={W[name]:.2f})", fontsize=10)
for ax in axes[1]: ax.set_xlabel("Year")
for ax in axes[:, 0]: ax.set_ylabel("mm / yr")
fig.suptitle("Individual model forecasts (median and 90% interval)"); plt.tight_layout(); plt.show()

## 6. Bayesian Model Averaging

The BMA predictive distribution is the weighted mixture of the model predictive distributions:

$$ p(\Delta \mid y) = \sum_k w_k\, p_k(\Delta \mid y, M_k) $$

Sampling: for each draw choose model *k* with probability *w<sub>k</sub>* and take one of its trajectories. Total variance decomposes as

$$ \operatorname{Var} = \underbrace{\sum_k w_k \sigma_k^2}_{\text{within-model}} + \underbrace{\sum_k w_k(\mu_k-\mu)^2}_{\text{between-model}} $$

In [ ]:
names = list(paths)
counts = rng.multinomial(N_SAMPLES, W[names].values)
bma = np.vstack([paths[n][rng.permutation(N_SAMPLES)[:c]] for n, c in zip(names, counts)])

def decompose(arr_by_model):
    mu_k  = np.array([arr_by_model[n].mean(axis=0) for n in names])
    var_k = np.array([arr_by_model[n].var(axis=0)  for n in names])
    w_ = W[names].values[:, None]
    mu = (w_ * mu_k).sum(0)
    return mu, (w_ * var_k).sum(0), (w_ * (mu_k - mu) ** 2).sum(0)

# annual rate
mu_r, within_r, between_r = decompose(paths)

# cumulative SLE from 1 Jan 2026 (mm)
cum_models = {n: np.cumsum(P, axis=1) for n, P in paths.items()}
cum_bma = np.cumsum(bma, axis=1)
mu_c, within_c, between_c = decompose(cum_models)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
lo, med, hi = np.percentile(bma, [5, 50, 95], axis=0)
lo50, hi50 = np.percentile(bma, [25, 75], axis=0)
ax = axes[0]
ax.plot(years, y, "ko-", ms=3, label="history")
ax.plot(fc_years, med, color="C3", label="BMA median")
ax.fill_between(fc_years, lo, hi, color="C3", alpha=0.18, label="90% interval")
ax.fill_between(fc_years, lo50, hi50, color="C3", alpha=0.30, label="50% interval")
ax.set(title="BMA forecast: annual glacier contribution", xlabel="Year", ylabel="mm / yr"); ax.legend()

lo, med, hi = np.percentile(cum_bma, [5, 50, 95], axis=0)
ax = axes[1]
ax.plot(fc_years, med, color="C3"); ax.fill_between(fc_years, lo, hi, color="C3", alpha=0.2)
for n in names:
    ax.plot(fc_years, np.median(cum_models[n], axis=0), lw=1, ls="--", label=n)
ax.set(title="Cumulative sea-level rise from 2026 (BMA 90% interval)", xlabel="Year", ylabel="mm"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].stackplot(fc_years, within_c, between_c, labels=["within-model", "between-model"], alpha=0.8)
axes[0].set(title="Variance of cumulative SLR: within vs between model", xlabel="Year", ylabel="mm$^2$"); axes[0].legend(loc="upper left")

axes[1].hist(cum_bma[:, -1], bins=50, density=True, alpha=0.7, color="C3")
for n in names:
    axes[1].axvline(cum_models[n][:, -1].mean(), ls="--", lw=1, label=n)
axes[1].set(title=f"Distribution of cumulative SLR by {FC_END} (BMA)", xlabel="mm"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 7. Summary of results

In [ ]:
def summ(c):
    e = c[:, -1]
    return pd.Series({"mean (mm)": e.mean(), "median (mm)": np.median(e), "5% (mm)": np.percentile(e, 5),
                      "95% (mm)": np.percentile(e, 95), "sd (mm)": e.std()})

table = pd.DataFrame({n: summ(c) for n, c in cum_models.items()}).T
table["BMA weight"] = W[table.index].values
table.loc["BMA"] = summ(cum_bma)
table.loc["BMA", "BMA weight"] = 1.0
table = table.round(2)
display(table)

share_between = between_c[-1] / (within_c[-1] + between_c[-1])
print(f"Cumulative glacier SLR {FC_START}–{FC_END} (BMA): {table.loc['BMA','mean (mm)']:.0f} mm "
      f"[90% interval {table.loc['BMA','5% (mm)']:.0f}–{table.loc['BMA','95% (mm)']:.0f} mm]")
print(f"Share of variance from model disagreement in {FC_END}: {share_between:.1%}")
print(f"Inventory ice ceiling: {total_scaling:.0f} mm SLE -> projected loss uses "
      f"{table.loc['BMA','mean (mm)']/total_scaling:.1%} of it (sanity check)")

out = pd.DataFrame({"year": fc_years,
                    "bma_rate_mean": bma.mean(0),
                    "bma_rate_p05": np.percentile(bma, 5, 0), "bma_rate_p95": np.percentile(bma, 95, 0),
                    "bma_cum_mean": cum_bma.mean(0),
                    "bma_cum_p05": np.percentile(cum_bma, 5, 0), "bma_cum_p95": np.percentile(cum_bma, 95, 0)})
out.to_csv("bma_forecast_2026_2056.csv", index=False)
print("Saved bma_forecast_2026_2056.csv")

## 8. Notes and limitations

* **The series is a placeholder unless `REAL_DATA_PATH` is set** (see the note at the top). Results above describe the method, not a real projection, until real observations are supplied.
* The inventory (maps from 1901–1998) provides the ice-volume ceiling only; it does not drive the regressions.
* Polynomial extrapolation over 30 years from ~26 points is risky. The validation weights show how much BMA trusts it.
* ARIMA samples condition on the estimated parameters, so its spread is a slight underestimate of its true uncertainty.
* Weights come from a short validation window (~9 forecasts), so they are themselves uncertain; compare `pseudo_bma` and `em` and try different `CV_HORIZON` values.
* Glacier-only: excludes the Greenland and Antarctic ice sheets, thermal expansion and land water storage.